# Additional Functions

The core RCPCHGrowth package functions are accessed through the `Measurement` class. The functions that make up the calculations that generate the `Measurement` class object though can be accessed independently as well some extras. 

## Date Calculations

There are four functions that can be leveraged:

`chronological_decimal_age`
`corrected_decimal_age`
`chronological_calendar_age`
`estimated_date_delivery`
`corrected_gestational_age`

In [1]:
# python imports
from datetime import date
import rcpchgrowth, sys

# set up
print(f"rcpchgrowth {rcpchgrowth.__version__} | Python {sys.version.split()[0]}")
%load_ext autoreload
%autoreload 2


birth_date = date(2020, 1, 1)
observation_date = date(2020, 1, 18)
gestation_weeks = 25
gestation_days = 4

chronological_age = rcpchgrowth.chronological_decimal_age(birth_date=birth_date, observation_date=observation_date)
corrected_age = rcpchgrowth.corrected_decimal_age(birth_date=birth_date, observation_date=observation_date, gestation_weeks=gestation_weeks, gestation_days=gestation_days)
edd = rcpchgrowth.estimated_date_delivery(birth_date=birth_date, gestation_weeks=gestation_weeks, gestation_days=gestation_days)
cga = rcpchgrowth.corrected_gestational_age(birth_date=birth_date, observation_date=observation_date, gestation_weeks=gestation_weeks, gestation_days=gestation_days)
chronological_age_calendar = rcpchgrowth.chronological_calendar_age(birth_date=birth_date, observation_date=observation_date)

print(f"Chronological Age (Decimal): {chronological_age:.3f} years")
print(f"Corrected Age (Decimal): {corrected_age:.3f} years")
print(f"Estimated Date of Delivery: {edd}")
print(f"Corrected Gestational Age: {cga.get('corrected_gestation_weeks')} + {cga.get('corrected_gestation_days')} days")
print(f"Chronological Age (Calendar): {chronological_age_calendar}")


rcpchgrowth 4.4.0 | Python 3.12.11
Chronological Age (Decimal): 0.047 years
Corrected Age (Decimal): -0.230 years
Estimated Date of Delivery: 2020-04-11
Corrected Gestational Age: 28 + 0 days
Chronological Age (Calendar): 2 weeks and 3 days


## BMI Functions

These functions calculate BMI but also include % median BMI

`bmi_from_height_weight`
`weight_for_bmi_height`
`percentage_median_bmi`

In [22]:
# python imports
from datetime import date

# RCPCHGrowth imports
import rcpchgrowth

birth_date = date(2010, 1, 1)
observation_date = date(2025, 1, 18)
height = 162
weight = 45

# Calculate BMI
bmi = rcpchgrowth.bmi_from_height_weight(height=height, weight=weight)

# Calculate age
age = rcpchgrowth.chronological_decimal_age(birth_date=birth_date, observation_date=observation_date)

print(f"BMI: {bmi:.2f} kg/m²")

# Calculate percentage median BMI
pct_median_bmi = rcpchgrowth.percentage_median_bmi(age=age, actual_bmi=bmi, sex="male", reference="uk-who")
print(f"Percentage Median BMI: {pct_median_bmi:.2f}%")

# Calculate the weight for a given height and BMI
target_bmi = 24
target_weight = rcpchgrowth.weight_for_bmi_height(bmi=target_bmi, height=height)
print(f"Target weight for BMI {target_bmi}: {target_weight:.2f} kg")


BMI: 17.15 kg/m²
Percentage Median BMI: 88.63%
Target weight for BMI 24: 62.99 kg


## Midparental Height Calculations

This calculates midparental height from parental heights. There are different methods and these are all supported.

Most paediatricians learn that midparental height is the average of the corrected height of the parents, where height is corrected for sex. It is reasoned that the average difference in height between men and women is 13 cm, therefore this is added to the height of the mother for a boy, or subtracted from the height of the father for a girl.

Published studies though show that this method is vulnerable to the 'regression to the mean' observation, where outliers (extremely tall or short people) tend to have children that are closer to the mean. It is this method that is endorsed by the RCPCH. The calculation here is to take the mean of the parental sds values and then apply a regression constant of 0.5. The final calculation is: (MatHtz +PatHtz)/4

| The strengths and limitations of parental heights as a predictor of attained height, Charlotte M Wright, Tim D Cheetham, Arch Dis Child 1999;81:257–260

In [30]:
from rcpchgrowth import *

paternal_height = 180
maternal_height = 165

standard_mid_parental_height = mid_parental_height(paternal_height=paternal_height, maternal_height=maternal_height, sex="male")

rcpch_mid_parental_height_z = mid_parental_height_z(paternal_height=paternal_height, maternal_height=maternal_height, reference="uk-who")
lower_threshold_z, upper_threshold_z = lower_and_upper_limits_of_expected_height_z(mid_parental_height_z=rcpch_mid_parental_height_z)

rcpch_mid_parental_height = measurement_from_sds(reference="uk-who", requested_sds=rcpch_mid_parental_height_z, age=20, sex="male", measurement_method="height")
lower_threshold = measurement_from_sds(reference="uk-who", requested_sds=lower_threshold_z, age=20, sex="male", measurement_method="height")
upper_threshold = measurement_from_sds(reference="uk-who", requested_sds=upper_threshold_z, age=20, sex="male", measurement_method="height")

print(f"Standard Mid-Parental Height: {standard_mid_parental_height:.2f} cm")
print(f"RCPCH Mid-Parental Height (Z-score): {rcpch_mid_parental_height_z:.2f}")
print(f"RCPCH Mid-Parental Height: {rcpch_mid_parental_height:.2f} cm")
print(f"Lower SDS Threshold (-1.4 SD): {lower_threshold:.2f} cm")
print(f"Upper SDS Threshold (+1.4 SD): {upper_threshold:.2f} cm")

Standard Mid-Parental Height: 179.00 cm
RCPCH Mid-Parental Height (Z-score): 0.15
RCPCH Mid-Parental Height: 178.40 cm
Lower SDS Threshold (-1.4 SD): 168.64 cm
Upper SDS Threshold (+1.4 SD): 188.15 cm
